In [ ]:
!pip install numpy matplotlib scikit-learn seaborn

# Lecture 07: Decision Trees

## Hierarchical Models for Classification and Regression

## Big Idea
Decision trees create a flowchart-like structure where each internal node represents a test on a feature, each branch represents the outcome of the test, and each leaf node represents a class label or continuous value.

## Everyday Metaphor

> 💡 **Decision Tree = playing 20 Questions**: Imagine trying to identify an animal by asking yes/no questions like 'Does it have fur?', 'Can it fly?', 'Does it live in water?'. Each question splits the possibilities until you can identify the animal.
> 
> **Features = the questions we ask**: Each feature in our data represents a potential question we can ask to split our data into more homogeneous groups.
> 
> **Tree depth = how specific we get**: A shallow tree gives broad categories; a deep tree gives very specific classifications but may overfit.

## Why It Matters
- Easy to interpret and visualize
- Handles both numerical and categorical data
- Requires little data preprocessing
- Can capture non-linear relationships and interactions
- Foundation for powerful ensemble methods like Random Forests and Gradient Boosting

## Formulas


### Decision Tree Basics

A decision tree consists of:
- **Root node**: Represents the entire dataset
- **Internal nodes**: Test a feature (e.g., "Age < 30?")
- **Branches**: Represent outcomes of tests (e.g., "Yes", "No")
- **Leaf nodes**: Represent final decisions (class labels or values)

### Key Concepts

- **Splitting criterion**: How to choose the best feature to split on
- **Pruning**: Techniques to prevent overfitting
- **Tree depth**: Controls model complexity
- **Leaf size**: Minimum samples required to make a leaf node


### Splitting Criteria for Classification

Common measures to determine the best split:

$$
\text{Entropy}(S) = -\sum_{i=1}^c p_i \log_2 p_i
$$

*Entropy: Measures impurity or disorder in a set*

$$
\text{Information Gain}(S, A) = \text{Entropy}(S) - \sum_{v \in \text{Values}(A)} \frac{|S_v|}{|S|} \text{Entropy}(S_v)
$$

*Information Gain: Reduction in entropy from splitting on attribute A*

$$
\text{Gini}(S) = 1 - \sum_{i=1}^c p_i^2
$$

*Gini Impurity: Probability of misclassifying a randomly chosen element*

*What each symbol means:*
- **S**: Dataset or subset
- **c**: Number of classes
- **p_i**: Proportion of class i in S
- **A**: Attribute/feature to split on
- **Values(A)**: Possible values of attribute A
- **S_v**: Subset of S where attribute A has value v


### Splitting Criteria for Regression

For regression trees, we minimize:

$$
\text{Variance}(S) = \frac{1}{|S|} \sum_{i=1}^{|S|} (y_i - \bar{y})^2
$$

*Variance: Measures spread of target values*

$$
\text{Reduction in Variance}(S, A) = \text{Variance}(S) - \sum_{v \in \text{Values}(A)} \frac{|S_v|}{|S|} \text{Variance}(S_v)
$$

*Reduction in Variance: How much splitting reduces variance*

*What each symbol means:*
- **S**: Dataset or subset
- **y_i**: Target value for sample i
- **\bar{y}**: Mean target value in S
- **A**: Attribute/feature to split on
- **Values(A)**: Possible values of attribute A
- **S_v**: Subset of S where attribute A has value v


## Worked Example by Hand

> 🧮 Simple decision tree: Classify these 2D points as blue or red:
Blue: [2,2], [2,3], [3,2], [8,8], [8,9], [9,8]
Red:  [2,8], [3,7], [4,6], [8,2], [9,1], [9,3]

### Step 1: Visualize the data

Points are clearly separable - blue points clustered in two groups, red points filling the gaps.

### Step 2: Choose first split (using Gini impurity)

Test splitting on x₁ at various thresholds:
- Split at x₁ = 4.5: Left: 6 blue, 3 red → Gini = 0.38
Right: 0 blue, 6 red → Gini = 0.0
Weighted Gini = (9/15)×0.38 + (6/15)×0.0 = 0.23

Test splitting on x₂ at various thresholds:
- Split at x₂ = 5.5: Bottom: 3 blue, 6 red → Gini = 0.38
Top: 3 blue, 0 red → Gini = 0.0
Weighted Gini = (9/15)×0.38 + (6/15)×0.0 = 0.23

Both splits give same Gini improvement. Choose x₁ = 4.5.

### Step 3: Split the data

**Left branch (x₁ < 4.5)**: 6 blue, 3 red
  - Further split on x₂ = 5.5:   - Bottom: 0 blue, 3 red → Pure red leaf
  - Top: 6 blue, 0 red → Pure blue leaf

**Right branch (x₁ ≥ 4.5)**: 0 blue, 6 red → Pure red leaf

### Final Tree:
IF x₁ < 4.5 THEN
  IF x₂ < 5.5 THEN Class = Red
  ELSE Class = Blue
ELSE
  Class = Red

<em>Answer: The decision tree perfectly separates the two classes with just two splits.</em>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree
from sklearn.datasets import make_classification, make_regression, load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
import pandas as pd


In [ ]:
# Decision Tree Classification
# Create a simple classification dataset
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,                            n_informative=2, n_clusters_per_class=1,                            random_state=42)

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train decision tree
clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

# Visualize the decision tree
plt.figure(figsize=(20, 10))
plot_tree(clf, feature_names=['Feature 1', 'Feature 2'],           class_names=['Class 0', 'Class 1'],           filled=True, rounded=True, fontsize=10)
plt.title('Decision Tree Classification (max_depth=3)')
plt.show()

# Also show text representation
tree_rules = export_text(clf, feature_names=['Feature 1', 'Feature 2'])
print("\nTree rules:")
print(tree_rules)

In [ ]:
# Decision Tree Regression
# Create a simple regression dataset
X, y = make_regression(n_samples=200, n_features=1, noise=10, random_state=42)

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train decision tree regressor
reg = DecisionTreeRegressor(max_depth=4, random_state=42)
reg.fit(X_train, y_train)

# Make predictions
y_pred = reg.predict(X_test)
mse = mean_squared_error(y_test, y_pred)

print(f"Mean Squared Error: {mse:.4f}")

# Visualize results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X, y, alpha=0.6)
plt.scatter(X_test, y_pred, color='red', alpha=0.8, s=30)
plt.xlabel('Feature')
plt.ylabel('Target')
plt.title('Decision Tree Regression')
plt.legend(['Training Data', 'Test Predictions'])
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs Predicted')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Decision Tree on Iris Dataset
# Load iris dataset
iris = load_iris()
X, y = iris.data, iris.target

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train decision tree
clf_iris = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_iris.fit(X_train, y_train)

# Evaluate
y_pred = clf_iris.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Iris dataset accuracy: {accuracy:.4f}")

# Visualize the tree
plt.figure(figsize=(15, 10))
plot_tree(clf_iris, feature_names=iris.feature_names,           class_names=iris.target_names,           filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree on Iris Dataset')
plt.show()

## Watch Out: Common Mistakes & Exam Traps

> ⚠️ **Common mistakes to avoid:**
> 1. **Overfitting**: Decision trees can easily overfit, especially when deep; always use pruning, max_depth, or min_samples_leaf to control complexity.
> 2. **Instability**: Small changes in data can lead to completely different trees; consider ensemble methods for more stable predictions.
> 3. **Bias toward features with more levels**: Features with many distinct values may be preferred in splitting; consider encoding or using entropy-based measures.
> 4. **Ignoring feature scaling**: While decision trees don't require scaling, some implementations may be affected by very different scales.
> 5. **Creating uninterpretable trees**: Very deep trees become impossible to interpret; balance accuracy with interpretability.
> 6. **Overlooking that trees create axis-aligned splits**: Decision trees can only split on single features at a time; diagonal boundaries require multiple splits.
> 7. **Neglecting to handle missing values**: Different implementations handle missing values differently; check your specific library's behavior.
> 8. **Using accuracy alone for evaluation**: Consider precision, recall, F1-score, especially for imbalanced datasets.


## Experiment Link

> 🔬 Run **Section 2: Decision Tree Experiment** in this notebook to explore how decision tree parameters affect model performance and visualize different tree structures.

This section will include:
- Exploration of max_depth, min_samples_split, min_samples_leaf parameters
- Visualization of decision boundaries for different tree complexities
- Comparison of entropy vs Gini splitting criteria
- Analysis of overfitting as tree depth increases
- Feature importance analysis
- Comparison with linear models to show when trees excel


## Videos to Watch

> 📺 **Recommended video searches:**

1. **Decision Tree Classification & Regression**
   - Search: https://www.youtube.com/results?search_query=decision+tree+machine+learning
   - Watch for: Clear explanation of how decision trees work for both classification and regression

2. **StatQuest: Decision Trees Part 1 - Basics**
   - Search: https://www.youtube.com/results?search_query=StatQuest+Decision+Trees
   - Watch for: Intuitive breakdown of entropy, information gain, and tree building

3. **Machine Learning Zoomcamp: Trees and Ensemble Methods**
   - Search: https://www.youtube.com/results?search_query=trees+ensemble+methods+machine+learning+zoomcamp
   - Watch for: Practical implementation and connection to Random Forests, Gradient Boosting


## Practice Questions

> ❓ Try to solve these yourself first, then check the answers below!

### Question 1 (Concept): What is the main advantage of decision trees over linear models?

<details>
<summary>Click to reveal answer</summary>

Decision trees can capture non-linear relationships and feature interactions without requiring explicit feature engineering, while linear models can only capture linear relationships unless interaction terms are manually added.

</details>


### Question 2 (Concept): How does entropy measure impurity in a dataset?

<details>
<summary>Click to reveal answer</summary>

Entropy measures the disorder or impurity in a dataset. A pure dataset (all same class) has entropy of 0. A perfectly balanced dataset (50/50 split between two classes) has entropy of 1. Higher entropy indicates more mixed classes.

</details>


### Question 3 (Calculation): Given a dataset with 4 samples of class A and 1 sample of class B, what is the Gini impurity?

<details>
<summary>Click to reveal answer</summary>

p_A = 4/5 = 0.8, p_B = 1/5 = 0.2
Gini = 1 - (p_A² + p_B²) = 1 - (0.8² + 0.2²) = 1 - (0.64 + 0.04) = 1 - 0.68 = 0.32

</details>


### Question 4 (Concept): What is the purpose of pruning in decision trees?

<details>
<summary>Click to reveal answer</summary>

Pruning reduces the size of decision trees by removing sections that provide little power to classify instances, which helps prevent overfitting and improves generalization to unseen data.

</details>


### Question 5 (Explain it simply): When would you prefer a decision tree over a neural network?

<details>
<summary>Click to reveal answer</summary>

Prefer decision trees when: you need interpretable models, you have small to medium datasets, you want fast training and prediction, you have mixed data types (numerical and categorical), or you need to explain predictions to stakeholders.

</details>


## Cheat Sheet Draft

> 📝 This section is optimized for printing - it should fit approximately 1/7 of an A4 page when printed.

- Optimized for printing - fits approximately 1/7 of an A4 page
- Contains key formulas, definitions, and common mistakes
- Designed for quick review and reference

### Key Formulas

$$
\text{Entropy}(S) = -\sum_{i=1}^c p_i \log_2 p_i
$$

*Entropy*

$$
\text{Information Gain}(S, A) = \text{Entropy}(S) - \sum_{v \in \text{Values}(A)} \frac{|S_v|}{|S|} \text{Entropy}(S_v)
$$

*Information Gain*

$$
\text{Gini}(S) = 1 - \sum_{i=1}^c p_i^2
$$

*Gini Impurity*

$$
\text{Variance}(S) = \frac{1}{|S|} \sum_{i=1}^{|S|} (y_i - \bar{y})^2
$$

*Variance*

### Key Definitions

- **Root Node:** Top node of the tree representing the entire dataset
- **Decision Node:** Internal node that tests a feature
- **Leaf Node:** Terminal node that represents a class label or value
- **Splitting Criterion:** Method used to choose the best feature to split on
- **Pruning:** Technique to reduce tree size and prevent overfitting
- **Feature Importance:** Measure of how much each feature contributes to tree purity
- **Max Depth:** Maximum number of levels in the tree
- **Min Samples Leaf:** Minimum samples required to be at a leaf node
### Common Mistakes to Avoid

- Overfitting by making trees too deep without pruning
- Ignoring the instability of decision trees (small data changes → big tree changes)
- Bias toward features with many distinct values
- Creating overly complex trees that are impossible to interpret
- Overlooking that decision trees create axis-aligned decision boundaries
- Neglecting to handle missing values appropriately
- Using a single accuracy metric without considering precision/recall for imbalanced data
